In [0]:
# Helpers
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

SCHEMA = "alpenia_dev.bronze"
tbl = lambda name: f"{SCHEMA}.{name}"

REPORT = []  # collected for the written report


def null_counts(df, cols):
    """NULL count per column, in a single pass."""
    row = df.agg(*[F.sum(F.col(c).isNull().cast("int")).alias(c) for c in cols]).first()
    return {c: int(row[c] or 0) for c in cols}


def check_and_save(name, raw, typed, cols_to_check):
    """Compare raw vs typed (row count + NULLs), then overwrite the table."""
    n_before, n_after = raw.count(), typed.count()
    nb = null_counts(raw, [c for c in cols_to_check if c in raw.columns])
    na = null_counts(typed, cols_to_check)

    print(f"\n=== {name} ===")
    print(f"rows: {n_before} -> {n_after}  {'OK' if n_before == n_after else 'MISMATCH'}")
    for c in cols_to_check:
        b = nb.get(c)  # None = column did not exist before (derived / flattened)
        a = na[c]
        status = "new column" if b is None else ("OK" if a == b else f"UNEXPECTED +{a - b} NULL")
        print(f"  {c:<22} NULL before={b}  after={a}  -> {status}")

    assert n_before == n_after, f"{name}: row count changed"

    # A failed conversion returns NULL instead of raising: stop before overwriting the table
    unexpected = {c: na[c] - nb[c] for c in cols_to_check
                  if nb.get(c) is not None and na[c] > nb[c]}
    assert not unexpected, f"{name}: unexpected NULLs after conversion {unexpected}"

    (typed.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")   # the schema changes (new types)
        .saveAsTable(tbl(name)))

    REPORT.append((name, n_before, n_after))

In [0]:
raw = spark.table(tbl("brz_contracts"))

# dd/MM/yyyy is not ISO -> explicit format (skip if already a DATE, for re-runs)
start_date = (F.to_date("start_date", "dd/MM/yyyy")
              if isinstance(raw.schema["start_date"].dataType, StringType)
              else F.col("start_date").cast("date"))

typed = raw.select(
    F.col("contract_id").cast("string"),
    F.col("client_id").cast("string"),
    F.col("product").cast("string"),
    F.col("city").cast("string"),
    start_date.alias("start_date"),
    F.col("annual_premium_eur").cast("double"),
    F.col("status").cast("string"),
)

check_and_save("brz_contracts", raw, typed,
               ["contract_id", "client_id", "start_date", "annual_premium_eur", "status"])

In [0]:
raw = spark.table(tbl("brz_clients"))

typed = raw.select(
    F.col("client_id").cast("string"),
    F.col("signup_date").cast("date"),        # ISO yyyy-MM-dd expected
    F.col("city").cast("string"),
    F.col("segment").cast("string"),
    F.col("birth_year").cast("int"),
)

check_and_save("brz_clients", raw, typed, ["client_id", "signup_date", "birth_year"])

In [0]:
raw = spark.table(tbl("brz_pricing"))

typed = raw.select(
    F.col("city").cast("string"),
    F.col("product").cast("string"),
    F.col("base_premium_eur").cast("decimal(7,2)"),
    F.col("rate_per_1k_eur").cast("decimal(5,2)"),
    F.col("valid_from").cast("date"),
)

check_and_save("brz_pricing", raw, typed,
               ["base_premium_eur", "rate_per_1k_eur", "valid_from"])

In [0]:
raw = spark.table(tbl("brz_claims"))

typed = raw.select(
    F.col("claim_id").cast("string"),
    F.col("contract_id").cast("string"),
    F.col("client_id").cast("string"),
    F.col("declared_ts").cast("timestamp"),
    F.col("settled_ts").cast("timestamp"),
    F.col("peril").cast("string"),
    F.col("amount_eur").cast("double"),
    F.col("deductible_eur").cast("double"),
).withColumn(
    # fractional days (datediff would only give whole days), rounded to 1 decimal
    "settlement_days",
    F.round((F.col("settled_ts").cast("long") - F.col("declared_ts").cast("long")) / 86400, 1),
)

check_and_save("brz_claims", raw, typed,
               ["declared_ts", "settled_ts", "amount_eur", "deductible_eur", "settlement_days"])

In [0]:
raw = spark.table(tbl("brz_web_events"))

# Check the real field names of the struct before flattening
if "quote" in raw.columns:
    print(raw.schema["quote"].dataType.simpleString())

event_ts = F.to_timestamp("event_ts")   # ISO 8601 with 'Z' is parsed natively

if "quote" in raw.columns:
    typed = raw.select(
        F.col("event_id").cast("string"),
        F.col("client_id").cast("string"),
        event_ts.alias("event_ts"),
        F.col("quote.product").cast("string").alias("quote_product"),
        F.col("quote.premium_eur").cast("double").alias("quote_premium_eur"),  # adjust if the field is named differently
        F.col("channel").cast("string"),
        F.col("action").cast("string"),
    )
else:  # already flattened (re-run)
    typed = raw.select(
        "event_id", "client_id", event_ts.alias("event_ts"),
        "quote_product", "quote_premium_eur", "channel", "action",
    )

check_and_save("brz_web_events", raw, typed,
               ["event_id", "client_id", "event_ts", "quote_product", "quote_premium_eur"])

assert "quote" not in spark.table(tbl("brz_web_events")).columns

In [0]:
raw = spark.table(tbl("brz_expertises"))

typed = raw.select(
    F.col("expertise_id").cast("string"),
    F.col("claim_id").cast("string"),
    F.to_timestamp("opened_at").alias("opened_at"),
    F.to_timestamp("closed_at").alias("closed_at"),   # NULL = still open
    F.col("finding_type").cast("string"),
    F.col("items_replaced").cast("array<string>"),
    F.col("cost_eur").cast("double"),
).withColumn(
    "items_count",
    F.when(F.col("items_replaced").isNull(), F.lit(0))   # robust to NULL arrays
     .otherwise(F.size("items_replaced"))
).withColumn(
    "is_open", F.col("closed_at").isNull()
)

check_and_save("brz_expertises", raw, typed,
               ["opened_at", "closed_at", "cost_eur", "items_count", "is_open"])

In [0]:
EXPECTED_ROWS = {
    "brz_contracts": 4000, "brz_clients": 3000, "brz_pricing": 6,
    "brz_claims": 24000, "brz_web_events": 90000, "brz_expertises": 450,
}
EXPECTED_TYPES = {
    "brz_contracts": {"contract_id": "string", "client_id": "string", "product": "string",
                      "city": "string", "start_date": "date",
                      "annual_premium_eur": "double", "status": "string"},
    "brz_clients": {"client_id": "string", "signup_date": "date", "city": "string",
                    "segment": "string", "birth_year": "int"},
    "brz_pricing": {"city": "string", "product": "string",
                    "base_premium_eur": "decimal(7,2)", "rate_per_1k_eur": "decimal(5,2)",
                    "valid_from": "date"},
    "brz_claims": {"claim_id": "string", "contract_id": "string", "client_id": "string",
                   "declared_ts": "timestamp", "settled_ts": "timestamp", "peril": "string",
                   "amount_eur": "double", "deductible_eur": "double",
                   "settlement_days": "double"},
    "brz_web_events": {"event_id": "string", "client_id": "string", "event_ts": "timestamp",
                       "quote_product": "string", "quote_premium_eur": "double",
                       "channel": "string", "action": "string"},
    "brz_expertises": {"expertise_id": "string", "claim_id": "string", "opened_at": "timestamp",
                       "closed_at": "timestamp", "finding_type": "string",
                       "items_replaced": "array<string>", "cost_eur": "double",
                       "items_count": "int", "is_open": "boolean"},
}

for name, exp_rows in EXPECTED_ROWS.items():
    df = spark.table(tbl(name))
    n = df.count()
    types_ok = dict(df.dtypes) == EXPECTED_TYPES[name]
    print(f"{name:<16} rows={n:>6} (expected {exp_rows:>6}) "
          f"{'OK' if n == exp_rows else 'KO'} | types {'OK' if types_ok else 'KO'}")
    if not types_ok:
        print("   actual:", dict(df.dtypes))